# CatBoost v2: подбор гиперпараметров

Задача этого ноутбука — улучшить модель при неизменном наборе `enhanced_v2` из 216 признаков. Мы не смешиваем изменение признаков и параметров модели, поэтому источник улучшения можно определить однозначно.

Подбор выполняется в два этапа:

1. быстрый screening глубины деревьев на раннем и позднем временных фолдах;
2. полная проверка победителя на всех четырёх holdout.

В этом первом запуске меняется только `depth`. Остальные параметры совпадают с CatBoost v2.

## 0. Режим запуска

`RUN_SCREENING=True` обучает новые конфигурации на двух фолдах. После выбора победителя `RUN_FULL_VALIDATION=True` запускает его на четырёх фолдах. Результаты сохраняются отдельно в `artifacts/catboost_tuning/`.

In [1]:
RUN_SCREENING = False
RUN_FULL_VALIDATION = False

SCREENING_ANCHORS = ('2025-10-22', '2026-01-14')

## 1. Импорты и данные

Используются готовые v2-срезы. Построение признаков здесь не повторяется.

In [2]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_TUNING_DIR,
    CATBOOST_V2_ARTIFACT_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    VALIDATION_ANCHORS,
    ensure_output_dirs,
)
from src.experiments import run_temporal_experiment
from src.features import load_snapshots
from src.models import make_validation_model
from src.validation import feature_columns, make_temporal_folds, rmsle

ensure_output_dirs()
snapshots = load_snapshots(CATBOOST_V2_SNAPSHOT_DIR, kind='train')
historical_anchors = sorted(snapshots)
features = feature_columns(snapshots[historical_anchors[0]])
assert len(features) == 216

all_folds = make_temporal_folds(historical_anchors, VALIDATION_ANCHORS)
screening_folds = [
    fold for fold in all_folds if fold.validation_anchor.isoformat() in SCREENING_ANCHORS
]
print(f'Признаков: {len(features)}')
print('Screening holdout:', [fold.validation_anchor for fold in screening_folds])

Признаков: 216
Screening holdout: [datetime.date(2025, 10, 22), datetime.date(2026, 1, 14)]


## 2. Кандидаты

Baseline v2 имеет глубину 8. Проверяем глубины 6, 7 и 9, не изменяя learning rate, регуляризацию и случайность. Такой one-factor-at-a-time эксперимент позволяет понять влияние именно глубины.

In [3]:
CANDIDATES = {
    'depth_6': {'depth': 6},
    'depth_7': {'depth': 7},
    'depth_9': {'depth': 9},
}

pd.DataFrame([
    {'candidate': name, **params} for name, params in CANDIDATES.items()
])

,candidate,depth
0,depth_6,6
1,depth_7,7
2,depth_9,9


## 3. Screening на двух временных фолдах

Берём самый ранний и самый поздний holdout. Это не заменяет полную валидацию, но позволяет не тратить время на четыре фолда для заведомо слабых конфигураций.

Depth 8 не переобучается: его уже рассчитанные метрики загружаются из эксперимента v2.

In [4]:
screening_path = CATBOOST_TUNING_DIR / 'depth_screening_metrics.csv'

if RUN_SCREENING:
    screening_parts = []
    for candidate_name, candidate_params in CANDIDATES.items():
        metrics, _ = run_temporal_experiment(
            snapshots=snapshots,
            folds=screening_folds,
            features=features,
            model_factory=partial(make_validation_model, **candidate_params),
            keep_oof=False,
            label=candidate_name,
        )
        screening_parts.append(metrics)
    screening_metrics = pd.concat(screening_parts, ignore_index=True)
    screening_metrics.to_csv(screening_path, index=False)
else:
    if not screening_path.exists():
        raise FileNotFoundError('Нет screening-метрик. Установите RUN_SCREENING=True.')
    screening_metrics = pd.read_csv(screening_path)

v2_metrics = pd.read_csv(CATBOOST_V2_ARTIFACT_DIR / 'multifold_metrics_default.csv')
depth_8_reference = v2_metrics[
    v2_metrics['validation_anchor'].isin(SCREENING_ANCHORS)
].copy()
depth_8_reference['experiment'] = 'depth_8_reference'
depth_8_reference['n_features'] = len(features)

screening_comparison = pd.concat(
    [screening_metrics, depth_8_reference[screening_metrics.columns]],
    ignore_index=True,
)
display(screening_comparison.sort_values(['validation_anchor', 'catboost_rmsle']))

,experiment,validation_anchor,n_features,n_train_rows,catboost_rmsle,baseline_rmsle,improvement,best_iteration
4,depth_9,2025-10-22,216,750000,1.714707,2.158052,0.443346,642
0,depth_6,2025-10-22,216,750000,1.714746,2.158052,0.443306,1302
2,depth_7,2025-10-22,216,750000,1.714748,2.158052,0.443304,713
6,depth_8_reference,2025-10-22,216,750000,1.714851,2.158052,0.443201,651
1,depth_6,2026-01-14,216,1500000,1.703598,2.195065,0.491467,722
5,depth_9,2026-01-14,216,1500000,1.704987,2.195065,0.490078,826
3,depth_7,2026-01-14,216,1500000,1.705430,2.195065,0.489635,991
7,depth_8_reference,2026-01-14,216,1500000,1.708149,2.195065,0.486916,91


## 4. Выбор кандидата

Основной критерий screening — средний RMSLE двух дат. Дополнительно смотрим худший фолд и разницу относительно depth 8 на каждой дате.

In [5]:
screening_summary = (
    screening_comparison.groupby('experiment', as_index=False)
    .agg(
        mean_rmsle=('catboost_rmsle', 'mean'),
        worst_rmsle=('catboost_rmsle', 'max'),
        mean_best_iteration=('best_iteration', 'mean'),
    )
    .sort_values('mean_rmsle')
)
display(screening_summary)

screening_winner = screening_summary.iloc[0]['experiment']
print('Победитель screening:', screening_winner)

,experiment,mean_rmsle,worst_rmsle,mean_best_iteration
0,depth_6,1.709172,1.714746,1012.0
3,depth_9,1.709847,1.714707,734.0
1,depth_7,1.710089,1.714748,852.0
2,depth_8_reference,1.711500,1.714851,371.0


Победитель screening: depth_6


## 5. Полная четырёхфолдовая проверка победителя

Победа на двух датах недостаточна для изменения финальной модели. Лучший новый depth проверяется на всех четырёх фолдах и сравнивается с depth 8.

Если screening выиграл исходный depth 8, дополнительное обучение не запускается.

In [6]:
full_metrics_path = CATBOOST_TUNING_DIR / 'depth_winner_multifold_metrics.csv'
full_oof_path = CATBOOST_TUNING_DIR / 'depth_winner_oof.parquet'

if RUN_FULL_VALIDATION and screening_winner != 'depth_8_reference':
    winner_params = CANDIDATES[screening_winner]
    full_metrics, full_oof = run_temporal_experiment(
        snapshots=snapshots,
        folds=all_folds,
        features=features,
        model_factory=partial(make_validation_model, **winner_params),
        keep_oof=True,
        label=screening_winner,
    )
    full_metrics.to_csv(full_metrics_path, index=False)
    full_oof.to_parquet(full_oof_path, index=False)
elif screening_winner == 'depth_8_reference':
    full_metrics = v2_metrics.copy()
    full_oof = pd.read_parquet(CATBOOST_V2_ARTIFACT_DIR / 'oof_predictions_default.parquet')
elif full_metrics_path.exists() and full_oof_path.exists():
    full_metrics = pd.read_csv(full_metrics_path)
    full_oof = pd.read_parquet(full_oof_path)
else:
    full_metrics = None
    full_oof = None
    print('Полная проверка ещё не запущена. После screening установите RUN_FULL_VALIDATION=True.')

## 6. Решение по параметру depth

После полной проверки новый depth принимается только при улучшении глобального OOF и отсутствии существенного провала на отдельном фолде. Финальный submission здесь не создаётся: подбор остальных параметров продолжится следующими блоками этого ноутбука.

In [7]:
if full_metrics is not None:
    depth_comparison = v2_metrics[[
        'validation_anchor', 'catboost_rmsle'
    ]].rename(columns={'catboost_rmsle': 'depth_8_rmsle'}).merge(
        full_metrics[['validation_anchor', 'catboost_rmsle', 'best_iteration']],
        on='validation_anchor',
        how='inner',
    ).rename(columns={'catboost_rmsle': 'candidate_rmsle'})
    depth_comparison['improvement'] = (
        depth_comparison['depth_8_rmsle'] - depth_comparison['candidate_rmsle']
    )
    display(depth_comparison)

    if 'prediction' in full_oof.columns:
        winner_oof_rmsle = rmsle(
            full_oof['target'].to_numpy(), full_oof['prediction'].to_numpy()
        )
    else:
        winner_oof_rmsle = rmsle(
            full_oof['target'].to_numpy(), full_oof['catboost_pred'].to_numpy()
        )
    print(f'Глобальный OOF RMSLE победителя: {winner_oof_rmsle:.6f}')
    print(f'Среднее улучшение по фолдам: {depth_comparison["improvement"].mean():.6f}')

,validation_anchor,depth_8_rmsle,candidate_rmsle,best_iteration,improvement
0,2025-10-22,1.714851,1.714746,1302,0.000105
1,2025-11-19,1.752544,1.752591,1315,-0.000047
2,2025-12-17,1.752768,1.752772,1234,-0.000003
3,2026-01-14,1.708149,1.703598,722,0.004551


Глобальный OOF RMSLE победителя: 1.731068
Среднее улучшение по фолдам: 0.001151


## Итог screening

На holdout 22.10 и 14.01 получены следующие средние RMSLE:

| Конфигурация | Средний RMSLE | Худший RMSLE |
|---|---:|---:|
| depth 6 | 1.709172 | 1.714746 |
| depth 9 | 1.709847 | 1.714707 |
| depth 7 | 1.710089 | 1.714748 |
| depth 8, v2 reference | 1.711500 | 1.714851 |

На раннем фолде различия малы, но на январском depth 6 улучшил RMSLE с `1.708149` до `1.703598`. Поэтому depth 6 был выбран для полной четырёхфолдовой проверки.

## Итог полной проверки depth=6

| Валидационный якорь | depth=8, v2 | depth=6 | Улучшение | Лучшая итерация depth=6 |
|---|---:|---:|---:|---:|
| 2025-10-22 | 1.714851 | 1.714746 | +0.000105 | 1302 |
| 2025-11-19 | 1.752544 | 1.752591 | −0.000047 | 1315 |
| 2025-12-17 | 1.752768 | 1.752772 | −0.000003 | 1234 |
| 2026-01-14 | 1.708149 | 1.703598 | +0.004551 | 722 |

Глобальный OOF RMSLE снизился с `1.732202` до `1.731068`, то есть на `0.001134`. Среднее улучшение по четырём фолдам равно `0.001151`. Результат воспроизводится на полной временной валидации, поэтому `depth=6` принимается как новый рабочий кандидат. При этом выигрыш почти полностью обеспечен январским фолдом, а на ноябрьском и декабрьском различия находятся около нуля.

Флаги запуска выше после эксперимента выключены: обычный запуск ноутбука только загрузит сохранённые результаты. Следующими независимыми гипотезами будут `l2_leaf_reg`, `random_strength` и `rsm`; они не смешиваются в одном сравнении.